# Lab 07-02: Compare Delegation With Agents as Tools

**Book chapter:** Chapter 07: Orchestrating Multi-Agent Systems  
**Estimated time:** 20 minutes

## Overview

Cymbal Retail's assistant must route returns and product questions to specialists. In
this lab, you build a front desk that hands the conversation to a specialist and a
concierge that consults specialists as tools, and you compare how each handles the
same two-part question.

## Learning Objectives

By the end of this lab, you will be able to:

- Delegate to sub-agents with transfer_to_agent
- Call agents as tools with AgentTool
- Explain who controls the conversation in each design
- Write descriptions that route requests reliably

## Before You Begin

- Complete **Lab 00: Set Up the Lab Environment** in the `Appendix-C-Setup` folder

**Cost:** A few cents for Gemini requests. No Google Cloud resources are created.

## How to Use This Notebook

Run the cells in order, from top to bottom, by selecting each code cell and pressing **Shift+Enter**. Read the explanation above each cell before you run it, and compare your output with the expected result that the text describes.

## Key Concepts

| Design | Who talks to the user | Best for |
|---|---|---|
| **Delegation** (`sub_agents`) | The specialist, after a transfer | Handing a conversation to an expert |
| **Agent as a tool** (`AgentTool`) | The calling agent | Combining several experts in one answer |

## Setup

In [ ]:
%pip install --quiet --upgrade google-adk

> **Note:** If you are using Google Colab and a message asks you to restart the
> session after the installation, click **Restart session**, and then continue
> from the next cell. You don't need to run the installation cell again.

In [ ]:
import os

# Replace with your lab project ID (see Appendix C).
PROJECT_ID = "your-project-id"  # @param {type:"string"}
# A current Gemini Flash model ID. Check the Google models page if this
# model has been retired.
MODEL = "gemini-3.8-flash"  # @param {type:"string"}

# Fall back to the environment variables set by labenv.sh (Appendix C).
if PROJECT_ID == "your-project-id":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", PROJECT_ID)
MODEL = os.environ.get("GEMINI_MODEL", MODEL)
assert PROJECT_ID != "your-project-id", "Set PROJECT_ID to your project ID."
# The lab code reads these environment variables.
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GEMINI_MODEL"] = MODEL
print("Project:", PROJECT_ID)
print("Model:  ", MODEL)

In [ ]:
import sys

if "google.colab" in sys.modules:
    # In Colab, sign in with the Google account that owns the lab project.
    from google.colab import auth
    auth.authenticate_user(project_id=PROJECT_ID)
    print("Signed in to Google Cloud from Colab.")
else:
    # In Jupyter, VS Code, or Workbench, the client libraries use Application
    # Default Credentials. If a later cell reports a credentials error, run
    # this command in a terminal and then rerun the cell:
    #   gcloud auth application-default login
    print("Using Application Default Credentials.")

## Step 1: Define two specialist agents

The two specialists each have one tool and a clear **description**, which is what a
coordinator reads when it decides where to route a request. Each specialist is created
by a function because an agent can have only one parent.

In [ ]:
import os

from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools.agent_tool import AgentTool
from google.genai import types

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
MODEL = os.environ.get("GEMINI_MODEL", "gemini-3.8-flash")


def check_return(order_id: str) -> dict:
    """Checks whether an order can still be returned."""
    days = {"10422": 12, "10423": 45}.get(order_id)
    if days is None:
        return {"order_id": order_id, "eligible": False,
                "reason": "Order not found"}
    return {"order_id": order_id, "eligible": days <= 30,
            "days_since_delivery": days}


def lookup_product(product: str) -> dict:
    """Returns sizing and care advice for a product."""
    return {"product": product, "sizing": "Runs half a size small",
            "care": "Reapply waterproofing every three months"}


# An agent can have only one parent, so each coordinator gets new copies.
def returns_agent():
    return Agent(name="returns_agent", model=MODEL, tools=[check_return],
                 description="Handles return requests and checks whether "
                             "an order can be returned.",
                 instruction="Check returns with check_return and explain "
                             "the result to the shopper.")


def product_agent():
    return Agent(name="product_agent", model=MODEL, tools=[lookup_product],
                 description="Answers questions about product sizing and "
                             "care.",
                 instruction="Answer with lookup_product and keep answers "
                             "short.")

## Step 2: Build a coordinator that delegates with sub_agents

The front desk has the specialists as **sub-agents**, so it can transfer the conversation to one of them with the built-in `transfer_to_agent` function.

In [ ]:
front_desk = Agent(
    name="front_desk", model=MODEL,
    instruction="You are Cymbal Retail's front desk. Transfer each request "
                "to the specialist whose description best matches it.",
    sub_agents=[returns_agent(), product_agent()])

## Step 3: Build a coordinator that calls specialists as tools

The concierge has the specialists as **tools**. It calls them like functions, keeps control of the conversation, and writes the final answer itself.

In [ ]:
concierge = Agent(
    name="concierge", model=MODEL,
    instruction="You answer shoppers yourself. Call the specialist tools "
                "to gather facts, then write one combined answer.",
    tools=[AgentTool(returns_agent()), AgentTool(product_agent())])

## Step 4: Send the same request to both and compare who answers

Send the same two-part question to both coordinators.

In [ ]:
def run(agent, text):
    """Runs one turn and shows the author and function calls per event."""
    runner = Runner(app_name="cymbal", agent=agent,
                    auto_create_session=True,
                    session_service=InMemorySessionService())
    print(f"\n=== {agent.name}: {text}")
    message = types.Content(role="user", parts=[types.Part(text=text)])
    for event in runner.run(user_id="shopper", session_id="s1",
                            new_message=message):
        for call in event.get_function_calls():
            print(f"  {event.author} calls {call.name}({dict(call.args)})")
        if event.is_final_response() and event.content and \
                event.content.parts:
            print(f"  Final answer from {event.author}: "
                  f"{event.content.parts[0].text}")


REQUEST = ("Can I return the boots from order 10422? Also, do the "
           "Trailhead boots run small?")
run(front_desk, REQUEST)
run(concierge, REQUEST)

**What to look for:**

| | front_desk | concierge |
|---|---|---|
| Function call | `transfer_to_agent` | `returns_agent`, `product_agent` |
| Final answer from | The specialist | The concierge |

## Step 5: Experiment

In Step 1, change the description of `product_agent` to `Helps shoppers.`, rerun Steps 1
to 4, and notice how vague descriptions make routing less reliable.

## Check Your Understanding

**1. Why does the front desk answer only part of the two-part question?**

<details><summary>Show answer</summary>

Control passes to a single specialist, which answers only the part it handles.

</details>

**2. When would you choose AgentTool over sub-agents?**

<details><summary>Show answer</summary>

When one agent must gather answers from several specialists and combine them into a single response.

</details>

**3. What does a coordinator use to decide where to transfer a request?**

<details><summary>Show answer</summary>

The descriptions of its sub-agents, which is why they must be clear and distinct.

</details>

## Summary

You built the same assistant with delegation and with agents as tools, and saw how each
design decides who talks to the shopper. This lab created no resources.